In [ ]:
!nvidia-smi

Mon Oct 26 16:29:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
from datasets import load_dataset

dataset = load_dataset("trl-lib/Capybara", split="train")

print("Dataset size:", len(dataset))
print("Columns:", dataset.column_names)
print("First example:", dataset[0])

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  482kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset size: 15806
Columns: ['source', 'messages', 'num_turns']
First example: {'source': 'GPT4LLM', 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbins p

In [ ]:
print("Rows:", len(dataset))
print("Columns:", dataset.column_names)
print("\nFirst conversation:")
print(dataset[0])

Rows: 15806
Columns: ['source', 'messages', 'num_turns']

First conversation:
{'source': 'GPT4LLM', 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'}, {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.', 'role': 'assistant'}, {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".', 'role': 'user'}, {'content': 'In "The Shawshank Redemption", Tim Robbins pla

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("Model loaded successfully!")
print("GPU memory allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!
GPU memory allocated: 3.86 GB


In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("QLoRA ready!")

trainable params: 6,815,744 || all params: 7,254,839,296 || trainable%: 0.0939
QLoRA ready!


In [ ]:
from datasets import DatasetDict

def format_conversation(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

formatted_dataset = dataset.map(
    format_conversation,
    remove_columns=dataset.column_names
)

split_dataset = formatted_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print("Training examples:", len(split_dataset["train"]))
print("Validation examples:", len(split_dataset["test"]))
print("\nSample:\n", split_dataset["train"][0]["text"][:500])

Map:   0%|          | 0/15806 [00:00<?, ? examples/s]

Training examples: 14225
Validation examples: 1581

Sample:
 <s>[INST] What do you know about VALORANT?[/INST] VALORANT is a free-to-play, multiplayer first-person shooter video game developed and published by Riot Games, the same company known for its popular game League of Legends. VALORANT was officially released on June 2, 2020, after having a closed beta period that started in April 2020.

The game is set in a near-future version of Earth and features agents with unique abilities, representing various countries and cultures. It is a team-based tactic


In [ ]:
from transformers import DataCollatorForLanguageModeling

# Start with a manageable subset
train_data = split_dataset["train"].select(
    range(min(2000, len(split_dataset["train"])))
)

val_data = split_dataset["test"].select(
    range(min(200, len(split_dataset["test"])))
)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

# Convert text into tokens
def tokenize_data(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )

tokenized_train = train_data.map(
    tokenize_data,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_data.map(
    tokenize_data,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Tokenization complete!")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Training examples: 2000
Validation examples: 200
Tokenization complete!


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./mistral-7b-assistant",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

print("Training configured successfully!")

Training configured successfully!


In [ ]:
trainer.train()

Step,Training Loss
10,1.279241
20,1.197890
30,1.227359
40,1.202997
50,1.128875
60,1.142850
70,1.058059
80,1.091598
90,1.153085
100,0.987844


TrainOutput(global_step=250, training_loss=1.0902382621765137, metrics={'train_runtime': 3867.7154, 'train_samples_per_second': 0.517, 'train_steps_per_second': 0.065, 'total_flos': 3.677650082694758e+16, 'train_loss': 1.0902382621765137, 'epoch': 1.0})

In [ ]:
import os

save_path = "/content/mistral-7b-assistant-final"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Adapter saved successfully!")
print("Files:", os.listdir(save_path))

Adapter saved successfully!
Files: ['tokenizer.json', 'tokenizer_config.json', 'README.md', 'chat_template.jinja', 'adapter_model.safetensors', 'adapter_config.json']


In [ ]:
from peft import PeftModel

model.eval()

messages = [
    {"role": "user", "content": "Hello! Who are you, and what can you help me with?"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Hello! I'm a helpful assistant who can assist you with various tasks. I can help you with information, answer your questions, guide you through a process, or even just chat about a topic of interest. What can I help you with today?

Disclaimer: I'm here to provide information and guidance, but I don't have personal experiences or feelings. I'm designed to help you, not the other way around. If you have any concerns or questions about my responses, please let me know.

If you're not sure where to start, feel free to ask me about a topic you're interested in, or ask me a question about something you're curious about. I'm here


In [ ]:
from google.colab import files
import shutil

shutil.make_archive(
    "/content/mistral-7b-assistant-final",
    "zip",
    "/content/mistral-7b-assistant-final"
)

files.download("/content/mistral-7b-assistant-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from huggingface_hub import login



In [ ]:
model.push_to_hub("mohidali9679/mistral-7b-assistant")
tokenizer.push_to_hub("mohidali9679/mistral-7b-assistant")

README.md:   0%|          | 0.00/819 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         |  556kB / 27.3MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/830 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/mohidali9679/mistral-7b-assistant/commit/e4d26111e843fbee3889292448a0b0c7686a6360', commit_message='Upload tokenizer', commit_description='', oid='e4d26111e843fbee3889292448a0b0c7686a6360', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mohidali9679/mistral-7b-assistant', endpoint='https://huggingface.co', repo_type='model', repo_id='mohidali9679/mistral-7b-assistant'), pr_revision=None, pr_num=None)

In [ ]:
!pip install -q fastapi uvicorn pyngrok

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="max-width:650px;margin:auto;font-family:Arial;background:#111827;color:white;padding:25px;border-radius:15px">
  <h2 style="text-align:center">🤖 Mistral AI Assistant</h2>
  <div id="chat" style="height:350px;overflow-y:auto;background:#1f2937;padding:15px;border-radius:10px;margin-bottom:15px">
    <p>🤖 Hello! I'm your AI assistant. How can I help?</p>
  </div>
  <input id="msg" placeholder="Type your message..." style="width:72%;padding:12px;border-radius:8px;border:0">
  <button onclick="sendMsg()" style="padding:12px;border:0;border-radius:8px;cursor:pointer">Send</button>
  <script>
    async function sendMsg(){
      const input=document.getElementById('msg');
      const chat=document.getElementById('chat');
      const message=input.value.trim();
      if(!message)return;
      chat.innerHTML+='<p><b>You:</b> '+message.replace(/</g,'&lt;').replace(/>/g,'&gt;')+'</p>';
      input.value='';
      chat.innerHTML+='<p><b>Assistant:</b> Connect the API in the next step.</p>';
      chat.scrollTop=chat.scrollHeight;
    }
  </script>
</div>
"""))

In [ ]:
%%writefile /content/main.py
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import torch

app = FastAPI(title="Mistral AI Assistant API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    message: str

@app.get("/")
def home():
    return {"status": "online", "message": "Mistral AI Assistant API"}

@app.post("/chat")
def chat(request: ChatRequest):
    messages = [{"role": "user", "content": request.message}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return {"reply": answer}

Writing /content/main.py


In [ ]:
import os

print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Saved model files:")

for path in [
    "/content/mistral-7b-assistant-final",
    "/content/mistral-7b-assistant"
]:
    if os.path.exists(path):
        print(f"\n{path}")
        print(os.listdir(path))

Model loaded: True
Tokenizer loaded: True
Saved model files:

/content/mistral-7b-assistant-final
['tokenizer.json', 'tokenizer_config.json', 'README.md', 'chat_template.jinja', 'adapter_model.safetensors', 'adapter_config.json']

/content/mistral-7b-assistant
['checkpoint-250']


In [ ]:
!uvicorn main:app --host 0.0.0.0 --port 8000

INFO:     Started server process [22536]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [22536]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "uvloop/loop.pyx", line 1509, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1502, in uvloop.loop.Loop.run_until_complete
    self.run_forever()
  File "uvloop/loop.pyx", line 1371, in uvloop.loop.Loop.run_forever
    self._run(mode)
  File "uvloop/loop.pyx", line 551, in uvloop.loop.Loop._run
    raise self._last_error
  File "uvloop/loop.pyx", line 471, in uvloop.loop.Loop._on

In [ ]:
import main
import threading
import uvicorn
import time

main.model = model
main.tokenizer = tokenizer

config = uvicorn.Config(main.app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

print("Server started:", server.started)

INFO:     Started server process [495]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server started: True


In [ ]:
print("model exists:", "model" in globals())
print("tokenizer exists:", "tokenizer" in globals())

model exists: True
tokenizer exists: True


In [ ]:
import gradio as gr
import torch


def sanitize_text(text):
    if isinstance(text, list):
        extracted = []
        for item in text:
            if isinstance(item, dict) and "text" in item:
                extracted.append(item["text"])
            else:
                extracted.append(str(item))
        return " ".join(extracted)
    return str(text)

def respond(message, history):
    messages = []

    # 1. Safely build history
    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role", "user")
            content = sanitize_text(turn.get("content", ""))
            messages.append({"role": role, "content": content})
        elif isinstance(turn, (list, tuple)):
            messages.append({"role": "user", "content": sanitize_text(turn[0])})
            if len(turn) > 1 and turn[1] is not None:
                messages.append({"role": "assistant", "content": sanitize_text(turn[1])})

    # 2. Add new user message
    messages.append({"role": "user", "content": sanitize_text(message)})

    try:
        # 3. Generate response
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
            )

        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    except Exception as e:

        return f"⚠️ Error: {str(e)}\n\nDebug Info: {str(messages)}"

gr.ChatInterface(respond, title="Mistral Assistant").launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8074658ee71947d8fc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"message": "Hello! What can you do?"}
)

print(response.status_code)
print(response.json())

200
{'reply': "Hello! As a helpful assistant, I can provide information, answer questions, and offer advice on a wide range of topics. I can help you with tasks such as writing, editing, and proofreading, as well as offer suggestions for problem-solving and decision-making. I can also assist with scheduling, reminders, and other organizational tasks. Let me know how I can help you today!\n\nHere's a list of things I can do:\n\n1. Provide information on a wide range of topics\n2. Answer questions\n3. Offer advice on various subjects\n4. Write and edit content\n5. Proofread and correct errors\n6. Suggest solutions to problems\n7"}
